# Task

**Tensors with NumPy.**
- Create the tensor

```python
T = np.arange(24).reshape(2, 3, 4)
```

and check that `T[i, j, k]` equals `T.reshape(-1)[12*i + 4*j + k]` for all indices, as in Example (see `ex:reshape` in the [lecture notes](https://github.com/ArystanK/ml_school_math/blob/main/Lecture1/lecture1.pdf)). Compute the unfoldings `T.reshape(2, 12)` and `T.transpose(1, 0, 2).reshape(3, 8)`. What are their ranks?
- Compute all the operations of Table (see `tab:einsum` in the [lecture notes](https://github.com/ArystanK/ml_school_math/blob/main/Lecture1/lecture1.pdf)) with `np.einsum` for random arrays, and compare with `@`, `np.trace`, `np.outer` and `.T`. For a batch of $100$ random $4 \times 4$ matrices, stored as an array of shape $100 \times 4 \times 4$, compute all their traces and all the products $A_bB_b$ with a single call of `np.einsum` each, and compare with a Python loop.
- For random matrices $A$ ($3 \times 4$), $B$ ($2 \times 5$) and $X$ ($4 \times 5$), check that `(A @ X @ B.T).reshape(-1)` equals `np.kron(A, B) @ X.reshape(-1)` (Remark (see `rem:kronecker` in the [lecture notes](https://github.com/ArystanK/ml_school_math/blob/main/Lecture1/lecture1.pdf))). How many numbers does `np.kron(A, B)` store for $A, B \in \mathbb{R}^{100 \times 100}$, and why is it much cheaper to compute $AXB^\top$ directly?
- Write a function that returns the outer product $\mathbf{u} \otimes \mathbf{v} \otimes \mathbf{w}$ of three vectors:

```python
def rank_one(u, v, w):
    return np.einsum("i,j,k->ijk", u, v, w)
```

For random vectors $\mathbf{u}$, $\mathbf{v}$ and $\mathbf{w}$, check that the slices `T[:, :, k]` of `T = rank_one(u, v, w)` are multiples of the matrix $\mathbf{u}\mathbf{v}^\top$ (Example (see `ex:tensor-rank-2` in the [lecture notes](https://github.com/ArystanK/ml_school_math/blob/main/Lecture1/lecture1.pdf))).


In [1]:
import numpy as np

np.set_printoptions(precision=5, suppress=True)
rng = np.random.default_rng(42)

In [2]:
def rref(A, tol=1e-10):
    """Return the reduced row echelon form and zero-based pivot columns."""
    R = np.array(A, dtype=float, copy=True)
    if R.ndim != 2:
        raise ValueError("A must be a two-dimensional array")
    m, n = R.shape
    pivots = []
    row = 0
    for col in range(n):
        if row == m:
            break
        pivot = row + np.argmax(np.abs(R[row:, col]))
        if abs(R[pivot, col]) < tol:
            continue
        R[[row, pivot]] = R[[pivot, row]]
        R[row] /= R[row, col]
        for i in range(m):
            if i != row:
                R[i] -= R[i, col] * R[row]
        pivots.append(col)
        row += 1
    R[np.abs(R) < tol] = 0.0
    return R, pivots


def rank(A, tol=1e-10):
    return len(rref(A, tol)[1])


In [3]:
# Check the indices and unfolding ranks.
T = np.arange(24).reshape(2, 3, 4)
print("Index formula holds:", all(T[i, j, k] == T.reshape(-1)[12*i + 4*j + k]
           for i in range(2) for j in range(3) for k in range(4)))
unfold1 = T.reshape(2, 12)
unfold2 = T.transpose(1, 0, 2).reshape(3, 8)
print("First unfolding:\n", unfold1, "\nrank =", rank(unfold1))
print("Second unfolding:\n", unfold2, "\nrank =", rank(unfold2))



Index formula holds: True
First unfolding:
 [[ 0  1  2  3  4  5  6  7  8  9 10 11]
 [12 13 14 15 16 17 18 19 20 21 22 23]] 
rank = 2
Second unfolding:
 [[ 0  1  2  3 12 13 14 15]
 [ 4  5  6  7 16 17 18 19]
 [ 8  9 10 11 20 21 22 23]] 
rank = 2


In [4]:
# Compare einsum with NumPy operations.
A = rng.normal(size=(4, 4))
B = rng.normal(size=(4, 4))
x, y = rng.normal(size=(2, 4))
checks = {
    "matrix product": (np.einsum("ij,jk->ik", A, B), A @ B),
    "trace": (np.einsum("ii->", A), np.trace(A)),
    "dot product": (np.einsum("i,i->", x, y), x @ y),
    "outer product": (np.einsum("i,j->ij", x, y), np.outer(x, y)),
    "transpose": (np.einsum("ij->ji", A), A.T),
    "bilinear form": (np.einsum("i,ij,j->", x, A, y), x @ A @ y),
}
batch_A = rng.normal(size=(100, 4, 4))
batch_B = rng.normal(size=(100, 4, 4))
checks["batch products"] = (np.einsum("bij,bjk->bik", batch_A, batch_B),
                            np.array([a @ b for a, b in zip(batch_A, batch_B)]))
checks["batch traces"] = (np.einsum("bii->b", batch_A),
                          np.array([np.trace(a) for a in batch_A]))
for name, (actual, expected) in checks.items():
    print(name, ":", np.allclose(actual, expected))

matrix product : True
trace : True
dot product : True
outer product : True
transpose : True
bilinear form : True
batch products : True
batch traces : True


In [5]:
# Check the Kronecker product identity.
A = rng.normal(size=(3, 4))
B = rng.normal(size=(2, 5))
X = rng.normal(size=(4, 5))
left = (A @ X @ B.T).reshape(-1)
right = np.kron(A, B) @ X.reshape(-1)
print("Kronecker identity: maximum error =", np.max(np.abs(left-right)))

# For 100 x 100 matrices, A ? B stores 10^8 numbers.
# Direct multiplication takes ~2?10^6 operations instead of ~10^8.

Kronecker identity: maximum error = 8.881784197001252e-16


In [6]:
def rank_one(u, v, w):
    return np.einsum("i,j,k->ijk", u, v, w)

# Check the slices of the rank-one tensor.
u, v, w = rng.normal(size=3), rng.normal(size=4), rng.normal(size=5)
T = rank_one(u, v, w)
print("Slices match w[k] * outer(u, v):",
      [np.allclose(T[:, :, k], w[k] * np.outer(u, v)) for k in range(len(w))])

Slices match w[k] * outer(u, v): [True, True, True, True, True]
